# Stored execution evidence — RANZCR CLiP

This public evidence copy preserves every textual output cell supplied in `mle_final_submission.zip`.
The original notebook SHA-256 is `794fd62e492f48edccb1be4ac9fe47fbc0e99ff630d116b39f28c9a9297b69e7`. Embedded binary rich media is removed so
that competition data and generated images are not redistributed. The repository setup cell
now targets `Isso-W/Jiaozi` at `main`; stored metrics and submission receipts remain historical
outputs and are not presented as a fresh rerun of the current branch.


# Standalone MLE-STAR benchmark runner — RANZCR CLiP

This notebook keeps the same structure as the supplied seven-benchmark runner,
but runs only **RANZCR CLiP - Catheter and Line Position Challenge**.

It performs real OOF baseline/MLE-STAR comparisons, reconstructs the selected
ensemble, fits final models, predicts the Kaggle test set, validates the exact
`sample_submission.csv` schema, and packages the exact final ensemble for a
Kaggle code-competition hidden-test rerun.

Runtime optimizations preserve the experiment protocol: a pixel-equivalent local
128×128 PNG cache avoids repeatedly decoding and resizing the original large JPEGs;
DataLoader prefetch, AMP, selected-epoch early stopping, and Drive checkpoints allow
safe resume after a Colab disconnect. No folds, labels, metrics, or ensemble weights
are approximated by these optimizations.


In [1]:
# Reproducible repository setup for Colab.
import os
import shutil
import subprocess
import sys
from pathlib import Path

REPOSITORY = "https://github.com/Isso-W/Jiaozi.git"
BRANCH = "main"
REPOSITORY_ROOT = Path("/content/Jiaozi")
EXPERIMENT_ROOT = REPOSITORY_ROOT / "experiments" / "mlestar_kaggle_benchmarks"

if REPOSITORY_ROOT.exists():
    shutil.rmtree(REPOSITORY_ROOT)
subprocess.run(
    ["git", "clone", "--depth", "1", "--branch", BRANCH, REPOSITORY, str(REPOSITORY_ROOT)],
    check=True,
)
actual_commit = subprocess.check_output(
    ["git", "-C", str(REPOSITORY_ROOT), "rev-parse", "HEAD"], text=True
).strip()
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", f"{EXPERIMENT_ROOT}[vision,llm,kaggle,dev]"],
    check=True,
)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "--upgrade", "kaggle==2.2.2"],
    check=True,
)
os.chdir(EXPERIMENT_ROOT)
if str(EXPERIMENT_ROOT) not in sys.path:
    sys.path.insert(0, str(EXPERIMENT_ROOT))
print("Repository commit:", actual_commit)
subprocess.run([sys.executable, "-m", "mlestar.cli", "benchmarks"], check=True)


Repository commit: 01228290ba427023f3e3d386255558f50b21a1a6


CompletedProcess(args=['/usr/bin/python3', '-m', 'mlestar.cli', 'benchmarks'], returncode=0)

In [2]:
# Credentials and experiment controls.
# Add KAGGLE_API_TOKEN in Colab: left sidebar -> key icon -> Secrets.
try:
    from google.colab import userdata
except ImportError:
    userdata = None

def load_colab_secret(name):
    if userdata is None:
        return
    try:
        value = userdata.get(name)
    except Exception:
        value = None
    if value:
        os.environ[name] = value

load_colab_secret("KAGGLE_API_TOKEN")
load_colab_secret("HF_TOKEN")
if os.environ.get("HF_TOKEN"):
    os.environ["HUGGINGFACE_HUB_TOKEN"] = os.environ["HF_TOKEN"]
os.environ["HF_HOME"] = "/content/hf_cache"

# Exact preprocessing cache: faster repeated OOF/final passes, same resized pixels.
PREPARE_IMAGE_CACHE = True
CACHE_RESIZED_IMAGES_TO_DRIVE = False

# Run All executes these seven pipelines from top to bottom. Comment out a
# name only when intentionally resuming or debugging one task.
BENCHMARKS_TO_RUN = {"ranzcr_clip"}

SEEDS = (13, 29, 47)
IMAGE_SIZE = 128
# Epoch count is selected inside each training run from an inner validation
# split. MAX_EPOCHS is only a safety ceiling, not the chosen epoch count.
MAX_EPOCHS = 40
MIN_EPOCHS = 3
EARLY_STOPPING_PATIENCE = 5
INNER_VALIDATION_FRACTION = 0.10
BATCH_SIZE = 32
MAX_TRAIN_SAMPLES_PER_FOLD = 2000
NUM_WORKERS = 4
USE_AMP = True  # Set False for strict FP32; AMP can cause tiny numerical differences.

# RANZCR is a code-only competition. After the exact sample schema and IDs
# pass, package the trained MLE ensemble for a Kaggle hidden-test Notebook.
PACKAGE_KAGGLE_ASSETS = True
KAGGLE_ASSET_ZIP_NAME = "MLE_STAR_RANZCR_Kaggle_Assets.zip"
# Make genuine data/training/prediction failures visibly red in Colab.
# Kaggle submission rejection is captured as a result rather than raised.
STOP_ON_PIPELINE_FAILURE = True

# Persist OOF artifacts, selected epochs, final checkpoints and submissions.
# Data archives remain under /content and may need to be downloaded again.
PERSIST_RUNS_TO_GOOGLE_DRIVE = True
RESUME_EXISTING_RUNS = True
# Allows recovery of the older notebook's completed OOF artifacts, which
# used fixed 3 epochs. Such a submission is labelled legacy and must not be
# mixed with the new inner-selected-epoch OOF comparison.
ALLOW_LEGACY_FIXED_EPOCH_OOF_RESUME = True

LEGACY_RUNS_ROOT = Path("/content/mlestar-runs")
if PERSIST_RUNS_TO_GOOGLE_DRIVE and userdata is not None:
    from google.colab import drive
    drive.mount("/content/drive")
    RUNS_ROOT = Path("/content/drive/MyDrive/mlestar-runs")
    # Rescue artifacts made by the older notebook in this still-live VM.
    # If the VM was already reset, /content has no old files to recover.
    if LEGACY_RUNS_ROOT.is_dir():
        shutil.copytree(LEGACY_RUNS_ROOT, RUNS_ROOT, dirs_exist_ok=True)
else:
    RUNS_ROOT = LEGACY_RUNS_ROOT
RUNS_ROOT.mkdir(parents=True, exist_ok=True)
print("Kaggle token configured:", bool(os.environ.get("KAGGLE_API_TOKEN")))
print("Selected benchmarks:", sorted(BENCHMARKS_TO_RUN))
print("Persistent runs root:", RUNS_ROOT)


Mounted at /content/drive
Kaggle token configured: True
Selected benchmarks: ['ranzcr_clip']
Persistent runs root: /content/drive/MyDrive/mlestar-runs


In [3]:
# Reusable Kaggle competition fetcher from the original notebook, rewritten
# without notebook shell interpolation so failures retain stdout/stderr.
import zipfile

def fetch_kaggle_competition(slug: str, data_root: str | Path, marker_file: str) -> Path:
    root = Path(data_root)
    root.mkdir(parents=True, exist_ok=True)
    if (root / marker_file).exists():
        print("Dataset already present:", root)
        return root
    if not os.environ.get("KAGGLE_API_TOKEN"):
        raise RuntimeError(
            f"No {marker_file} in {root} and KAGGLE_API_TOKEN is not configured."
        )
    command = [
        "kaggle", "competitions", "download", "-c", slug,
        "-p", str(root),
    ]
    completed = subprocess.run(
        command, text=True, capture_output=True, check=False,
    )
    if completed.stdout.strip():
        print(completed.stdout.strip())
    if completed.returncode != 0:
        detail = completed.stderr.strip() or completed.stdout.strip() or "No error text returned"
        raise RuntimeError(
            f"Kaggle download failed for {slug} (exit {completed.returncode}).\n"
            f"Command: {' '.join(command)}\n"
            f"Kaggle response:\n{detail}"
        )
    outer_zip = root / f"{slug}.zip"
    if outer_zip.exists():
        with zipfile.ZipFile(outer_zip) as archive:
            archive.extractall(root)
    for nested_zip in root.glob("*.zip"):
        if nested_zip == outer_zip:
            continue
        with zipfile.ZipFile(nested_zip) as archive:
            archive.extractall(root)
    if not (root / marker_file).exists():
        raise RuntimeError(
            f"Download did not produce {marker_file} in {root}. Inspect the Kaggle CLI "
            "output and confirm that the token is current and rules are accepted."
        )
    return root


In [4]:
#@title RANZCR pixel-equivalent resize cache (shared helper) { display-mode: "form" }
import json
from concurrent.futures import ThreadPoolExecutor

import numpy as np
from PIL import Image
from tqdm.auto import tqdm

def prepare_ranzcr_cache(source_root: str | Path) -> Path:
    source_root = Path(source_root)
    if not PREPARE_IMAGE_CACHE:
        return source_root
    cache_root = (
        RUNS_ROOT / "_data_cache" / f"ranzcr_resized_{IMAGE_SIZE}"
        if CACHE_RESIZED_IMAGES_TO_DRIVE
        else Path(f"/content/ranzcr-resized-{IMAGE_SIZE}")
    )
    cache_root.mkdir(parents=True, exist_ok=True)
    for csv_name in ("train.csv", "sample_submission.csv"):
        source = source_root / csv_name
        if not source.is_file():
            raise FileNotFoundError(source)
        shutil.copy2(source, cache_root / csv_name)

    work = []
    for split in ("train", "test"):
        source_dir = source_root / split
        destination_dir = cache_root / split
        destination_dir.mkdir(parents=True, exist_ok=True)
        work.extend(
            (path, destination_dir / f"{path.stem}.png")
            for path in sorted(source_dir.glob("*.jpg"))
        )
    if not work:
        raise RuntimeError(f"No RANZCR JPEG images found in {source_root}")

    def resize_one(item):
        source, destination = item
        if destination.is_file():
            return False
        with Image.open(source) as opened:
            resized = opened.convert("RGB").resize((IMAGE_SIZE, IMAGE_SIZE))
            temporary = destination.with_suffix(".tmp.png")
            resized.save(temporary, format="PNG", compress_level=1)
            temporary.replace(destination)
        return True

    created = 0
    with ThreadPoolExecutor(max_workers=max(1, NUM_WORKERS)) as executor:
        for result in tqdm(
            executor.map(resize_one, work), total=len(work), desc="RANZCR exact resize cache"
        ):
            created += int(result)
    missing = [str(destination) for _, destination in work if not destination.is_file()]
    if missing:
        raise RuntimeError(f"Incomplete RANZCR cache, examples: {missing[:5]}")

    # Prove that the cache yields exactly the pixels used by the original adapter.
    for source, cached in work[:10]:
        with Image.open(source) as opened:
            expected = np.asarray(
                opened.convert("RGB").resize((IMAGE_SIZE, IMAGE_SIZE)), dtype=np.uint8
            )
        with Image.open(cached) as opened:
            actual = np.asarray(opened.convert("RGB"), dtype=np.uint8)
        if not np.array_equal(expected, actual):
            raise AssertionError(f"Cached pixels differ for {source.name}")
    print({"cached_images": len(work), "new_files": created, "pixel_checks": 10})
    return cache_root


In [5]:
#@title MLE-selected epoch vision adapter (shared helper) { display-mode: "form" }
# Shared optimized adapter for the RANZCR image benchmark.
from contextlib import nullcontext
from time import perf_counter

import numpy as np
import torch
import mlestar.adapters.vision as vision
import mlestar.experiment as experiment_module
from mlestar.experiment import compare

class FastVisionMixin:
    def __init__(
        self,
        *args,
        num_workers=4,
        use_amp=True,
        min_epochs=3,
        early_stopping_patience=5,
        inner_validation_fraction=0.10,
        max_inner_validation_samples=500,
        **kwargs,
    ):
        super().__init__(*args, **kwargs)
        self.num_workers = int(num_workers)
        self.use_amp = bool(use_amp and vision._DEVICE.type == "cuda")
        self.min_epochs = int(min_epochs)
        self.early_stopping_patience = int(early_stopping_patience)
        self.inner_validation_fraction = float(inner_validation_fraction)
        self.max_inner_validation_samples = int(max_inner_validation_samples)
        self._fit_number = 0
        self._epoch_records = []
        self._epoch_context = {}
        if not 1 <= self.min_epochs <= self.epochs:
            raise ValueError("min_epochs must be in [1, max_epochs]")
        if self.early_stopping_patience < 1:
            raise ValueError("early_stopping_patience must be positive")
        if not 0 < self.inner_validation_fraction < 0.5:
            raise ValueError("inner_validation_fraction must be in (0, 0.5)")

    def _loader(self, dataset, *, shuffle, seed=None, drop_last=False):
        options = dict(
            dataset=dataset,
            batch_size=self.batch_size,
            shuffle=shuffle,
            num_workers=self.num_workers,
            pin_memory=(vision._DEVICE.type == "cuda"),
            persistent_workers=(self.num_workers > 0),
            drop_last=bool(drop_last),
        )
        if self.num_workers > 0:
            options["prefetch_factor"] = 2
        if shuffle:
            options["generator"] = torch.Generator().manual_seed(int(seed))
        return torch.utils.data.DataLoader(**options)

    def _scaler(self):
        try:
            return torch.amp.GradScaler("cuda", enabled=self.use_amp)
        except (AttributeError, TypeError):
            return torch.cuda.amp.GradScaler(enabled=self.use_amp)

    def _autocast(self):
        if not self.use_amp:
            return nullcontext()
        return torch.autocast(device_type="cuda", dtype=torch.float16)

    def _fit(self, model, dataset, seed):
        parameters = list(model.parameters())
        if not parameters:
            return
        self._fit_number += 1
        if len(dataset) < 2:
            raise ValueError("Epoch selection requires at least two training rows")
        validation_size = min(
            max(1, int(round(len(dataset) * self.inner_validation_fraction))),
            self.max_inner_validation_samples,
        )
        validation_size = min(validation_size, len(dataset) - 1)
        permutation = torch.randperm(
            len(dataset), generator=torch.Generator().manual_seed(int(seed))
        ).tolist()
        validation_dataset = torch.utils.data.Subset(dataset, permutation[:validation_size])
        fitting_dataset = torch.utils.data.Subset(dataset, permutation[validation_size:])
        initial_state = {
            name: value.detach().cpu().clone()
            for name, value in model.state_dict().items()
        }
        loader = self._loader(
            fitting_dataset,
            shuffle=True,
            seed=seed,
            drop_last=(
                len(fitting_dataset) > self.batch_size
                and len(fitting_dataset) % self.batch_size == 1
            ),
        )
        validation_loader = self._loader(validation_dataset, shuffle=False)
        optimizer = torch.optim.Adam(parameters, lr=1e-4)
        loss_fn = self._loss_fn()
        scaler = self._scaler()
        best_loss = float("inf")
        best_epoch = None
        stale_epochs = 0
        for epoch in range(self.epochs):
            started = perf_counter()
            loss_sum = 0.0
            model.train()
            for images, targets in loader:
                images = images.to(vision._DEVICE, non_blocking=True)
                targets = targets.to(vision._DEVICE, non_blocking=True)
                optimizer.zero_grad(set_to_none=True)
                with self._autocast():
                    loss = loss_fn(model(images), targets)
                if not torch.isfinite(loss):
                    raise FloatingPointError(f"Non-finite loss: {loss.item()}")
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
                loss_sum += float(loss.detach().cpu())

            validation_loss_sum = 0.0
            validation_rows = 0
            model.eval()
            with torch.inference_mode():
                for images, targets in validation_loader:
                    images = images.to(vision._DEVICE, non_blocking=True)
                    targets = targets.to(vision._DEVICE, non_blocking=True)
                    with self._autocast():
                        validation_loss = loss_fn(model(images), targets)
                    batch_rows = int(images.shape[0])
                    validation_loss_sum += float(validation_loss.detach().cpu()) * batch_rows
                    validation_rows += batch_rows
            mean_validation_loss = validation_loss_sum / max(validation_rows, 1)
            improved = mean_validation_loss < best_loss - 1e-8
            if improved:
                best_loss = mean_validation_loss
                best_epoch = epoch + 1
                stale_epochs = 0
            else:
                stale_epochs += 1
            print(
                f"  fit={self._fit_number} epoch={epoch + 1}/{self.epochs} "
                f"loss={loss_sum / max(len(loader), 1):.5f} "
                f"inner_val_loss={mean_validation_loss:.5f} "
                f"best_epoch={best_epoch} "
                f"seconds={perf_counter() - started:.1f}"
            )
            if (
                epoch + 1 >= self.min_epochs
                and stale_epochs >= self.early_stopping_patience
            ):
                break
        if best_epoch is None:
            raise RuntimeError("Epoch selection did not produce a finite checkpoint")
        # The inner split chooses only the epoch count. Rewind the model and
        # train from the same initialization on every available training row,
        # leaving the outer OOF fold untouched.
        model.load_state_dict(initial_state)
        model.to(vision._DEVICE)
        torch.manual_seed(int(seed))
        if vision._DEVICE.type == "cuda":
            torch.cuda.manual_seed_all(int(seed))
        full_loader = self._loader(
            dataset,
            shuffle=True,
            seed=seed,
            drop_last=(len(dataset) > self.batch_size and len(dataset) % self.batch_size == 1),
        )
        full_optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
        full_scaler = self._scaler()
        for refit_epoch in range(best_epoch):
            model.train()
            full_loss_sum = 0.0
            for images, targets in full_loader:
                images = images.to(vision._DEVICE, non_blocking=True)
                targets = targets.to(vision._DEVICE, non_blocking=True)
                full_optimizer.zero_grad(set_to_none=True)
                with self._autocast():
                    full_loss = loss_fn(model(images), targets)
                if not torch.isfinite(full_loss):
                    raise FloatingPointError(f"Non-finite refit loss: {full_loss.item()}")
                full_scaler.scale(full_loss).backward()
                full_scaler.step(full_optimizer)
                full_scaler.update()
                full_loss_sum += float(full_loss.detach().cpu())
            print(
                f"  fit={self._fit_number} full_refit_epoch={refit_epoch + 1}/{best_epoch} "
                f"loss={full_loss_sum / max(len(full_loader), 1):.5f}"
            )
        record = {
            **self._epoch_context,
            "fit_number": self._fit_number,
            "seed": int(seed),
            "dataset_rows": len(dataset),
            "inner_fit_rows": len(fitting_dataset),
            "inner_validation_rows": len(validation_dataset),
            "selected_epoch": int(best_epoch),
            "best_inner_validation_loss": float(best_loss),
            "max_epochs": int(self.epochs),
            "stopped_early": bool(best_epoch < self.epochs),
            "criterion": "minimum inner-validation loss",
            "refit_on_all_training_rows": True,
        }
        self._epoch_records.append(record)
        self.artifacts.write_json(
            "epoch_selection.json",
            {"leakage_rule": "outer OOF rows are not used for epoch selection", "records": self._epoch_records},
        )

    def _predict(self, model, dataset):
        loader = self._loader(dataset, shuffle=False)
        outputs = []
        model.eval()
        with torch.inference_mode():
            for images, _ in loader:
                images = images.to(vision._DEVICE, non_blocking=True)
                with self._autocast():
                    outputs.append(self._predict_probs(model(images)))
        return np.concatenate(outputs, axis=0)

    def run(self, candidate, *, phase, seed, parent_experiment_id=None):
        model_name = candidate.block("model")
        print(f"START seed={seed} phase={phase} model={model_name}")
        started = perf_counter()
        self._epoch_context = {
            "phase": phase,
            "candidate_id": candidate.candidate_id,
            "model": model_name,
        }
        result = super().run(
            candidate, phase=phase, seed=seed,
            parent_experiment_id=parent_experiment_id,
        )
        print(
            f"END   seed={seed} phase={phase} model={model_name} "
            f"metric={result.receipt.metric_value} error={result.receipt.error} "
            f"minutes={(perf_counter() - started) / 60:.2f}"
        )
        self._epoch_context = {}
        return result

for benchmark, base_class in list(experiment_module.ADAPTER_CLASSES.items()):
    if issubclass(base_class, vision.ImageClassificationAdapter):
        fast_class = type(f"Fast{base_class.__name__}", (FastVisionMixin, base_class), {})
        experiment_module.ADAPTER_CLASSES[benchmark] = fast_class

image_requested = any(name != "leaf_classification" for name in BENCHMARKS_TO_RUN)
if image_requested and vision._DEVICE.type != "cuda":
    raise RuntimeError("Enable a Colab GPU runtime before running image benchmarks.")
print(
    "device:",
    torch.cuda.get_device_name(0) if vision._DEVICE.type == "cuda" else "CPU (leaf only)",
)


device: NVIDIA L4


In [6]:
#@title Benchmark runner (shared helper) { display-mode: "form" }
# Unified task registry and runner. Each original benchmark cell below calls
# this same function, so paths, seeds and adapter options cannot drift.
import json
import pandas as pd

RANZCR_TARGETS = (
    "ETT - Abnormal",
    "ETT - Borderline",
    "ETT - Normal",
    "NGT - Abnormal",
    "NGT - Borderline",
    "NGT - Incompletely Imaged",
    "NGT - Normal",
    "CVC - Abnormal",
    "CVC - Borderline",
    "CVC - Normal",
    "Swan Ganz Catheter Present",
)

TASKS = {
    "ranzcr_clip": {
        "slug": "ranzcr-clip-catheter-line-classification",
        "marker": "train.csv",
        "data_root": "/content/ranzcr-clip-catheter-line-classification",
        "vision": True,
    },
}

# The pinned repository has a generic image-multilabel pipeline but no RANZCR
# catalog entry. Register the contract and adapter in memory for this runtime.
from types import MappingProxyType
import benchmarks.catalog as catalog_module
from mlestar.contracts import FoldSpec, MetricSpec, SubmissionSpec, TaskSpec

ranzcr_task = TaskSpec(
    key="ranzcr_clip",
    competition="ranzcr-clip-catheter-line-classification",
    modality="image_multilabel",
    metric=MetricSpec("roc_auc"),
    fold=FoldSpec(n_splits=5),
    submission=SubmissionSpec(
        id_columns=("StudyInstanceUID",),
        prediction_columns=RANZCR_TARGETS,
        prediction_from_sample=False,
    ),
    target_columns=RANZCR_TARGETS,
    description="Eleven-label catheter and line position classification scored by mean ROC-AUC.",
)
catalog_module.BENCHMARKS = MappingProxyType(
    {**dict(catalog_module.BENCHMARKS), "ranzcr_clip": ranzcr_task}
)

class RanzcrAdapter(vision.ImageClassificationAdapter):
    def _load_dataset(self, data_root):
        frame = pd.read_csv(Path(data_root) / "train.csv")
        missing = sorted(set(RANZCR_TARGETS) - set(frame.columns))
        if missing:
            raise ValueError(f"RANZCR train.csv is missing target columns: {missing}")
        ids = frame["StudyInstanceUID"].astype(str).tolist()
        suffix = ".png" if (Path(data_root) / "train" / f"{ids[0]}.png").is_file() else ".jpg"
        paths = [Path(data_root) / "train" / f"{value}{suffix}" for value in ids]
        labels = frame[list(RANZCR_TARGETS)].to_numpy(dtype=float)
        return paths, labels, ids

class FastRanzcrAdapter(FastVisionMixin, RanzcrAdapter):
    pass

experiment_module.ADAPTER_CLASSES["ranzcr_clip"] = FastRanzcrAdapter
experiment_module._CANDIDATE_MODELS["ranzcr_clip"] = ("resnet18", "efficientnet_b0")

COMPLETED = {}

def run_benchmark(benchmark):
    if benchmark not in BENCHMARKS_TO_RUN:
        print(f"SKIP {benchmark}; add it to BENCHMARKS_TO_RUN to execute.")
        return None
    config = TASKS[benchmark]
    source_root = fetch_kaggle_competition(
        config["slug"], config["data_root"], config["marker"]
    )
    data_root = prepare_ranzcr_cache(source_root) if benchmark == "ranzcr_clip" else source_root
    run_root = RUNS_ROOT / benchmark
    comparison_path = run_root / "comparison.csv"
    receipts_path = run_root / "receipts.jsonl"
    report_path = run_root / "comparison.json"
    if RESUME_EXISTING_RUNS and comparison_path.is_file() and receipts_path.is_file():
        comparison = pd.read_csv(comparison_path)
        expected_rows = {(seed, arm) for seed in SEEDS for arm in (
            "baseline", "mlestar_initial", "mlestar_refined", "mlestar_ensemble"
        )}
        actual_rows = set(zip(comparison["seed"].astype(int), comparison["arm"].astype(str)))
        if actual_rows != expected_rows:
            raise RuntimeError(
                f"Incomplete or incompatible cached comparison for {benchmark}; "
                f"expected {len(expected_rows)} seed/arm rows, got {len(actual_rows)}"
            )
        has_epoch_evidence = any(run_root.rglob("epoch_selection.json"))
        if not has_epoch_evidence and not ALLOW_LEGACY_FIXED_EPOCH_OOF_RESUME:
            raise RuntimeError(
                f"{benchmark} has legacy fixed-epoch OOF artifacts. Set "
                "ALLOW_LEGACY_FIXED_EPOCH_OOF_RESUME=True to reuse them, or delete "
                "the run directory to recompute with MLE-selected epochs."
            )
        protocol = "mle_inner_selected_epochs" if has_epoch_evidence else "legacy_fixed_3_epochs"
        report = (
            json.loads(report_path.read_text())
            if report_path.is_file()
            else {"benchmark": benchmark, "status": "resumed_from_artifacts"}
        )
        COMPLETED[benchmark] = {
            "data_root": str(data_root), "run_root": str(run_root),
            "report": report, "resumed": True, "oof_epoch_protocol": protocol,
        }
        print(f"RESUME {benchmark}: OOF comparison skipped ({protocol}).")
        display(comparison)
        display(comparison.groupby("arm")["metric_value"].agg(["mean", "std", "count"]))
        return report
    adapter_kwargs = {}
    if config["vision"]:
        adapter_kwargs = {
            "pretrained": True,
            "image_size": IMAGE_SIZE,
            "epochs": MAX_EPOCHS,
            "batch_size": BATCH_SIZE,
            "max_train_samples": MAX_TRAIN_SAMPLES_PER_FOLD,
            "num_workers": NUM_WORKERS,
            "use_amp": USE_AMP,
            "min_epochs": MIN_EPOCHS,
            "early_stopping_patience": EARLY_STOPPING_PATIENCE,
            "inner_validation_fraction": INNER_VALIDATION_FRACTION,
        }
    report = compare(
        benchmark=benchmark,
        data_root=data_root,
        run_root=run_root,
        seeds=SEEDS,
        outer_rounds=1,
        inner_rounds=1,
        adapter_kwargs=adapter_kwargs,
    )
    comparison = pd.read_csv(run_root / "comparison.csv")
    COMPLETED[benchmark] = {
        "data_root": str(data_root), "run_root": str(run_root), "report": report,
        "resumed": False, "oof_epoch_protocol": "mle_inner_selected_epochs",
    }
    display(comparison)
    display(comparison.groupby("arm")["metric_value"].agg(["mean", "std", "count"]))
    return report


In [7]:
#@title Kaggle prediction and submission (shared helper) { display-mode: "form" }
# Generic final prediction, validation, Kaggle API submission and status capture.
# Each dataset cell below calls run_complete_benchmark exactly once.
import json
import os
import subprocess
from io import StringIO
from pathlib import Path
from time import monotonic, sleep

import numpy as np
import pandas as pd
from timm import create_model
from benchmarks.catalog import get_task
from mlestar.ensemble import blend_test_predictions, select_ensemble

PIPELINE_RESULTS = {}

def build_mle_ensemble_plans(benchmark, data_root, run_root):
    task = get_task(benchmark)
    receipts = [
        json.loads(line)
        for line in (Path(run_root) / "receipts.jsonl").read_text().splitlines()
        if line.strip()
    ]
    if benchmark == "leaf_classification":
        labels = pd.read_csv(Path(data_root) / "train.csv")["species"].astype(str).to_numpy()
    else:
        _, _, labels, _ = vision_data(benchmark, data_root)
    plans = []
    for seed in SEEDS:
        def receipt_for(phase):
            matched = [
                row for row in receipts
                if row.get("phase") == phase and int(row.get("seed")) == seed
                and row.get("metric_value") is not None
            ]
            if len(matched) != 1:
                raise RuntimeError(
                    f"Expected one successful {phase} receipt for {benchmark}, seed {seed}"
                )
            return matched[0]

        baseline = receipt_for("baseline")
        initial_selected = receipt_for("initial_selected")
        refined = receipt_for("refined_selected")
        baseline_oof = pd.read_csv(Path(run_root) / f"seed_{seed}" / baseline["oof_path"])
        refined_oof = pd.read_csv(Path(run_root) / f"seed_{seed}" / refined["oof_path"])
        baseline_values = baseline_oof.iloc[:, 1:].to_numpy(dtype=float)
        refined_values = refined_oof.iloc[:, 1:].to_numpy(dtype=float)
        if baseline_values.shape[1] == 1:
            baseline_values = baseline_values[:, 0]
            refined_values = refined_values[:, 0]
        score_transform = None
        if task.modality == "image_ordinal":
            max_label = int(np.max(labels))
            score_transform = lambda prediction, bound=max_label: np.clip(
                np.rint(prediction), 0, bound
            )
        ensemble = select_ensemble(
            {
                "baseline": (baseline_oof.iloc[:, 0].astype(str).tolist(), baseline_values),
                "refined": (refined_oof.iloc[:, 0].astype(str).tolist(), refined_values),
            },
            labels,
            task.metric,
            score_transform=score_transform,
        )
        comparison = pd.read_csv(Path(run_root) / "comparison.csv")
        reported = comparison.loc[
            (comparison["seed"] == seed)
            & (comparison["arm"] == "mlestar_ensemble"),
            "metric_value",
        ]
        if len(reported) != 1 or not np.isclose(
            float(reported.iloc[0]), float(ensemble.score.value), rtol=0, atol=1e-12
        ):
            raise AssertionError(
                f"Reconstructed ensemble score does not match comparison.csv for seed {seed}"
            )
        plan = {
            "seed": seed,
            "weights": ensemble.weights,
            "oof_score": ensemble.score.value,
            "baseline_receipt": baseline,
            "refined_receipt": refined,
        }
        if benchmark != "leaf_classification":
            initial_id = str(initial_selected["candidate_id"])
            refined_id = str(refined["candidate_id"])
            if refined_id == initial_id:
                refined_model = initial_id
            elif refined_id == f"{initial_id}-model":
                refined_model = (
                    "efficientnet_b0" if initial_id == "resnet18" else "resnet18"
                )
            else:
                raise RuntimeError(
                    f"Cannot reconstruct refined model from initial={initial_id}, refined={refined_id}"
                )
            if refined_model not in {"resnet18", "efficientnet_b0"}:
                raise RuntimeError(f"Unexpected reconstructed model: {refined_model}")
            plan["baseline_model"] = "resnet18"
            plan["refined_model"] = refined_model
        plans.append(plan)
    display(pd.DataFrame([
        {
            "seed": plan["seed"],
            "baseline_weight": plan["weights"]["baseline"],
            "refined_weight": plan["weights"]["refined"],
            "refined_model": plan.get("refined_model", "receipt test predictions"),
            "reconstructed_oof_score": plan["oof_score"],
        }
        for plan in plans
    ]))
    return plans

def build_leaf_submission(data_root, run_root, plans):
    sample = pd.read_csv(Path(data_root) / "sample_submission.csv")
    id_column = sample.columns[0]
    prediction_frames = []
    for plan in plans:
        seed = plan["seed"]
        candidates = {}
        for name, receipt in (
            ("baseline", plan["baseline_receipt"]),
            ("refined", plan["refined_receipt"]),
        ):
            if not receipt.get("test_path"):
                raise RuntimeError(f"Leaf {name} receipt has no test predictions for seed {seed}")
            frame = pd.read_csv(Path(run_root) / f"seed_{seed}" / receipt["test_path"])
            if list(frame.columns) != list(sample.columns):
                raise ValueError("Leaf prediction columns differ from sample_submission.csv")
            if frame[id_column].astype(str).tolist() != sample[id_column].astype(str).tolist():
                raise ValueError("Leaf test IDs or order differ from sample_submission.csv")
            candidates[name] = frame.iloc[:, 1:].to_numpy(dtype=float)
        prediction_frames.append(blend_test_predictions(candidates, plan["weights"]))
    submission = pd.DataFrame(
        np.mean(np.stack(prediction_frames), axis=0),
        columns=list(sample.columns[1:]),
    )
    submission.insert(0, id_column, sample[id_column].to_numpy())
    return submission

def vision_data(benchmark, data_root):
    root = Path(data_root)
    sample = pd.read_csv(root / "sample_submission.csv")
    if benchmark == "plant_pathology_2020":
        train = pd.read_csv(root / "train.csv")
        labels = train[["healthy", "multiple_diseases", "rust", "scab"]].to_numpy(float)
        train_paths = [root / "images" / f"{value}.jpg" for value in train["image_id"].astype(str)]
        test_paths = [root / "images" / f"{value}.jpg" for value in sample["image_id"].astype(str)]
    elif benchmark == "aptos_2019":
        train = pd.read_csv(root / "train.csv")
        labels = train["diagnosis"].to_numpy(float)
        train_paths = [root / "train_images" / f"{value}.png" for value in train["id_code"].astype(str)]
        test_paths = [root / "test_images" / f"{value}.png" for value in sample["id_code"].astype(str)]
    elif benchmark == "dog_breed":
        train = pd.read_csv(root / "labels.csv")
        classes = list(sample.columns[1:])
        mapping = {name: index for index, name in enumerate(classes)}
        if train["breed"].map(mapping).isna().any():
            raise ValueError("Dog Breed labels contain a class absent from sample_submission.csv")
        labels = train["breed"].map(mapping).to_numpy(int)
        train_paths = [root / "train" / f"{value}.jpg" for value in train["id"].astype(str)]
        test_paths = [root / "test" / f"{value}.jpg" for value in sample["id"].astype(str)]
    elif benchmark == "aerial_cactus":
        train = pd.read_csv(root / "train.csv")
        labels = train["has_cactus"].to_numpy(float)
        train_paths = [root / "train" / value for value in train["id"].astype(str)]
        test_paths = [root / "test" / value for value in sample["id"].astype(str)]
    elif benchmark == "dogs_vs_cats":
        train_paths = sorted((root / "train").glob("*.jpg"))
        labels = np.asarray([1.0 if path.name.startswith("dog.") else 0.0 for path in train_paths])
        test_paths = [root / "test" / f"{value}.jpg" for value in sample["id"].astype(str)]
    elif benchmark == "histopathologic_cancer":
        train = pd.read_csv(root / "train_labels.csv")
        labels = train["label"].to_numpy(float)
        train_paths = [root / "train" / f"{value}.tif" for value in train["id"].astype(str)]
        test_paths = [root / "test" / f"{value}.tif" for value in sample["id"].astype(str)]
    elif benchmark == "ranzcr_clip":
        train = pd.read_csv(root / "train.csv")
        labels = train[list(RANZCR_TARGETS)].to_numpy(dtype=float)
        train_ids = train["StudyInstanceUID"].astype(str).tolist()
        test_ids = sample["StudyInstanceUID"].astype(str).tolist()
        suffix = ".png" if (root / "train" / f"{train_ids[0]}.png").is_file() else ".jpg"
        train_paths = [root / "train" / f"{value}{suffix}" for value in train_ids]
        test_paths = [root / "test" / f"{value}{suffix}" for value in test_ids]
    else:
        raise KeyError(benchmark)
    missing = [str(path) for path in train_paths + test_paths if not path.is_file()]
    if missing:
        raise FileNotFoundError(f"Missing {benchmark} images, examples: {missing[:5]}")
    return sample, train_paths, labels, test_paths

def build_vision_submission(benchmark, data_root, run_root, plans):
    task = get_task(benchmark)
    sample, train_paths, labels, test_paths = vision_data(benchmark, data_root)
    adapter_class = experiment_module.ADAPTER_CLASSES[benchmark]
    predictions = []
    checkpoint_root = Path(run_root) / "final_checkpoints"
    checkpoint_root.mkdir(parents=True, exist_ok=True)
    for plan in plans:
        seed = plan["seed"]
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        adapter = adapter_class(
            data_root, Path(run_root) / f"final_seed_{seed}", task,
            pretrained=True, image_size=IMAGE_SIZE, epochs=MAX_EPOCHS,
            batch_size=BATCH_SIZE, max_train_samples=None,
            num_workers=NUM_WORKERS, use_amp=USE_AMP,
            min_epochs=MIN_EPOCHS,
            early_stopping_patience=EARLY_STOPPING_PATIENCE,
            inner_validation_fraction=INNER_VALIDATION_FRACTION,
        )
        targets = adapter._prepare_targets(np.asarray(labels))
        train_dataset = vision._ImageDataset(train_paths, targets, IMAGE_SIZE)
        if task.modality == "image_multiclass":
            dummy = torch.zeros(len(test_paths), dtype=torch.long)
        elif task.modality == "image_multilabel":
            dummy = torch.zeros((len(test_paths), len(task.target_columns)), dtype=torch.float32)
        else:
            dummy = torch.zeros((len(test_paths), 1), dtype=torch.float32)
        test_dataset = vision._ImageDataset(test_paths, dummy, IMAGE_SIZE)
        num_classes = adapter._num_classes(np.asarray(labels))
        candidate_predictions = {}
        weighted_models = {}
        for arm_name in ("baseline", "refined"):
            weight = float(plan["weights"][arm_name])
            if weight > 0.0:
                model_name = plan[f"{arm_name}_model"]
                weighted_models[model_name] = weighted_models.get(model_name, 0.0) + weight
        if not weighted_models:
            raise RuntimeError(f"Seed {seed} has no positive MLE ensemble weight")
        for model_name in sorted(weighted_models):
            # Seed each candidate independently, matching the adapter's OOF
            # model construction instead of letting one model consume the
            # next model's random-number stream.
            torch.manual_seed(seed)
            torch.cuda.manual_seed_all(seed)
            model = create_model(model_name, pretrained=True, num_classes=num_classes).to(vision._DEVICE)
            checkpoint_path = checkpoint_root / f"{model_name}_seed_{seed}.pt"
            resumed = False
            if RESUME_EXISTING_RUNS and checkpoint_path.is_file():
                try:
                    checkpoint = torch.load(
                        checkpoint_path, map_location=vision._DEVICE, weights_only=False
                    )
                except TypeError:  # compatibility with older PyTorch
                    checkpoint = torch.load(checkpoint_path, map_location=vision._DEVICE)
                if checkpoint.get("model_name") != model_name or int(checkpoint.get("seed")) != int(seed):
                    raise ValueError(f"Checkpoint metadata mismatch: {checkpoint_path}")
                model.load_state_dict(checkpoint["model"])
                resumed = True
                print(f"Resume final checkpoint: benchmark={benchmark} seed={seed} model={model_name}")
            if not resumed:
                print(f"Final fit: benchmark={benchmark} seed={seed} model={model_name}")
                adapter._epoch_context = {
                    "phase": "final_full_data",
                    "candidate_id": model_name,
                    "model": model_name,
                }
                adapter._fit(model, train_dataset, seed)
                temporary_path = checkpoint_path.with_suffix(".tmp.pt")
                torch.save(
                    {"model": model.state_dict(), "model_name": model_name, "seed": seed},
                    temporary_path,
                )
                temporary_path.replace(checkpoint_path)
            candidate_predictions[model_name] = np.asarray(
                adapter._predict(model, test_dataset), dtype=np.float64
            )
            del model
            torch.cuda.empty_cache()
        predictions.append(sum(
            weight * candidate_predictions[model_name]
            for model_name, weight in weighted_models.items()
        ))
        torch.cuda.empty_cache()
    raw = np.stack(predictions, axis=0)
    mean_prediction = raw.mean(axis=0)
    submission = sample.copy()
    if task.modality == "image_ordinal":
        submission[task.submission.prediction_columns[0]] = np.clip(
            np.rint(mean_prediction), 0, int(np.max(labels))
        ).astype(int)
    elif mean_prediction.ndim == 1:
        submission[task.submission.prediction_columns[0]] = mean_prediction
    else:
        columns = list(sample.columns[1:]) if task.submission.prediction_from_sample else list(task.target_columns)
        if mean_prediction.shape != (len(sample), len(columns)):
            raise ValueError(
                f"Prediction shape {mean_prediction.shape} does not match {len(columns)} columns"
            )
        submission[columns] = mean_prediction
    np.save(Path(run_root) / "raw_test_predictions.npy", raw)
    return submission

def validate_submission(benchmark, data_root, submission):
    task = get_task(benchmark)
    sample = pd.read_csv(Path(data_root) / "sample_submission.csv")
    id_column = task.submission.id_columns[0]
    if list(submission.columns) != list(sample.columns):
        raise AssertionError(f"{benchmark}: columns differ from sample_submission.csv")
    if len(submission) != len(sample):
        raise AssertionError(f"{benchmark}: row count differs from sample_submission.csv")
    if submission[id_column].astype(str).tolist() != sample[id_column].astype(str).tolist():
        raise AssertionError(f"{benchmark}: IDs or order differ from sample_submission.csv")
    values = submission.drop(columns=[id_column]).to_numpy(dtype=float)
    if not np.isfinite(values).all():
        raise AssertionError(f"{benchmark}: predictions contain NaN or infinity")
    if task.modality == "image_ordinal":
        if not np.equal(values, np.rint(values)).all() or not ((values >= 0) & (values <= 4)).all():
            raise AssertionError("APTOS predictions must be integer grades 0..4")
    elif not ((values >= 0) & (values <= 1)).all():
        raise AssertionError(f"{benchmark}: probabilities are outside [0, 1]")
    print(f"Validated {benchmark} submission: rows={len(submission)}, columns={len(submission.columns)}")

def collect_epoch_selection(run_root):
    records = []
    for path in sorted(Path(run_root).rglob("epoch_selection.json")):
        payload = json.loads(path.read_text())
        for record in payload.get("records", []):
            records.append({"artifact": str(path.relative_to(run_root)), **record})
    if not records:
        return [], None
    frame = pd.DataFrame(records)
    output_path = Path(run_root) / "selected_epochs.csv"
    frame.to_csv(output_path, index=False)
    display(frame)
    display(
        frame.groupby(["phase", "model"])["selected_epoch"]
        .agg(["median", "mean", "min", "max", "count"])
    )
    print("Epoch-selection evidence:", output_path)
    return records, str(output_path)

def _sha256(path):
    import hashlib

    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def package_kaggle_assets(benchmark, run_root, plans, submission_path):
    """Package only trained final models and their exact ensemble contract.

    RANZCR is code-only: a normal CSV API upload cannot access the hidden test
    images. The returned ZIP is uploaded as a private Kaggle Dataset and used
    by the companion offline inference Notebook.
    """
    import timm

    record = {"requested": bool(PACKAGE_KAGGLE_ASSETS), "accepted": False}
    if not PACKAGE_KAGGLE_ASSETS:
        record["status"] = "disabled"
        return record
    if benchmark != "ranzcr_clip":
        raise ValueError(f"Kaggle asset packaging is only defined for RANZCR, got {benchmark}")

    run_root = Path(run_root)
    checkpoint_root = run_root / "final_checkpoints"
    asset_dir = run_root / "kaggle_assets"
    if asset_dir.exists():
        shutil.rmtree(asset_dir)
    asset_dir.mkdir(parents=True, exist_ok=True)

    serialized_plans = []
    copied_files = set()
    for plan in plans:
        seed = int(plan["seed"])
        weighted_models = {}
        for arm_name in ("baseline", "refined"):
            weight = float(plan["weights"][arm_name])
            if weight <= 0.0:
                continue
            model_name = str(plan[f"{arm_name}_model"])
            weighted_models[model_name] = weighted_models.get(model_name, 0.0) + weight
        total_weight = sum(weighted_models.values())
        if not np.isclose(total_weight, 1.0, rtol=0, atol=1e-10):
            raise AssertionError(f"Seed {seed} ensemble weights sum to {total_weight}, not 1")

        serialized_models = []
        for model_name, weight in sorted(weighted_models.items()):
            filename = f"{model_name}_seed_{seed}.pt"
            source_path = checkpoint_root / filename
            if not source_path.is_file():
                raise FileNotFoundError(f"Final checkpoint missing: {source_path}")
            destination = asset_dir / filename
            if filename not in copied_files:
                shutil.copy2(source_path, destination)
                copied_files.add(filename)
            serialized_models.append(
                {
                    "model_name": model_name,
                    "weight": float(weight),
                    "checkpoint_file": filename,
                    "sha256": _sha256(destination),
                }
            )
        serialized_plans.append(
            {
                "seed": seed,
                "oof_score": float(plan["oof_score"]),
                "models": serialized_models,
            }
        )

    public_submission = asset_dir / "public_test_submission.csv"
    shutil.copy2(submission_path, public_submission)
    manifest = {
        "format_version": 1,
        "competition": "ranzcr-clip-catheter-line-classification",
        "benchmark": benchmark,
        "repository_commit": PINNED_COMMIT,
        "image_size": int(IMAGE_SIZE),
        "batch_size": int(BATCH_SIZE),
        "id_column": "StudyInstanceUID",
        "target_columns": list(RANZCR_TARGETS),
        "preprocessing": "PIL RGB; resize exactly to image_size x image_size; float32 / 255; CHW",
        "prediction_activation": "sigmoid",
        "seed_aggregation": "arithmetic mean",
        "plans": serialized_plans,
        "torch_version": torch.__version__,
        "timm_version": timm.__version__,
        "public_test_rows": int(len(pd.read_csv(submission_path))),
    }
    manifest_path = asset_dir / "mle_star_ranzcr_manifest.json"
    manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")

    zip_path = RUNS_ROOT / KAGGLE_ASSET_ZIP_NAME
    temporary_zip = zip_path.with_suffix(".tmp.zip")
    if temporary_zip.exists():
        temporary_zip.unlink()
    with zipfile.ZipFile(temporary_zip, "w", compression=zipfile.ZIP_STORED, allowZip64=True) as archive:
        for path in sorted(asset_dir.iterdir()):
            if path.is_file():
                archive.write(path, arcname=path.name)
    temporary_zip.replace(zip_path)

    record.update(
        accepted=True,
        status="assets_ready",
        asset_dir=str(asset_dir),
        asset_zip=str(zip_path),
        asset_zip_mb=zip_path.stat().st_size / (1024 ** 2),
        manifest=str(manifest_path),
        model_files=sorted(copied_files),
        next_step=(
            "Upload this ZIP as a private Kaggle Dataset, attach it and the RANZCR "
            "competition data to MLE_STAR_RANZCR_CLiP_Kaggle_Inference.ipynb, "
            "then Save Version -> Save & Run All -> Submit."
        ),
    )
    print(json.dumps(record, indent=2, ensure_ascii=False))
    return record


def run_complete_benchmark(benchmark):
    result = {"benchmark": benchmark, "stage": "starting", "success": False}
    PIPELINE_RESULTS[benchmark] = result
    if benchmark not in BENCHMARKS_TO_RUN:
        result.update(stage="skipped", error=None)
        print(f"SKIP {benchmark}")
        return result
    try:
        result["stage"] = "data_download_and_oof_comparison"
        run_benchmark(benchmark)
        info = COMPLETED[benchmark]
        data_root = Path(info["data_root"])
        run_root = Path(info["run_root"])
        plans = build_mle_ensemble_plans(benchmark, data_root, run_root)
        result["oof_epoch_protocol"] = info.get("oof_epoch_protocol")
        result["resumed_oof"] = bool(info.get("resumed"))
        result["final_protocol"] = (
            "per-seed mlestar_ensemble OOF weights, then 3-seed mean; "
            + str(info.get("oof_epoch_protocol", "unknown epoch protocol"))
        )
        result["ensemble_plans"] = plans

        result["stage"] = "final_prediction"
        if benchmark == "leaf_classification":
            submission = build_leaf_submission(data_root, run_root, plans)
        else:
            submission = build_vision_submission(benchmark, data_root, run_root, plans)
        epoch_records, epoch_path = collect_epoch_selection(run_root)
        result["epoch_selection"] = {
            "method": (
                "not applicable to tree models"
                if benchmark == "leaf_classification"
                else "minimum loss on an inner split of each training fold"
            ),
            "evidence_path": epoch_path,
            "records": epoch_records,
        }
        validate_submission(benchmark, data_root, submission)
        submission_path = run_root / "submission.csv"
        submission.to_csv(submission_path, index=False)
        result["submission_path"] = str(submission_path)
        display(submission.head())

        result["stage"] = "kaggle_asset_packaging"
        result["kaggle"] = package_kaggle_assets(
            benchmark, run_root, plans, submission_path
        )
        result.update(stage="complete", success=True)
    except Exception as error:
        result.update(
            success=False,
            error=f"{type(error).__name__}: {error}",
            failed_stage=result["stage"],
            stage="failed",
        )
        print(f"FAILED {benchmark}: {result['error']}")
        if STOP_ON_PIPELINE_FAILURE:
            raise
    finally:
        result_path = RUNS_ROOT / f"{benchmark}_pipeline_result.json"
        result_path.write_text(json.dumps(result, indent=2, default=str), encoding="utf-8")
        print(json.dumps(
    {
        "benchmark": result.get("benchmark"),
        "success": result.get("success"),
        "submission_path": result.get("submission_path"),
        "latest_submission": result.get(
            "kaggle", {}
        ).get("latest_submission"),
        "error": result.get("error"),
    },
    indent=2,
    ensure_ascii=False,
    default=str,
))
    return result


In [8]:
# RANZCR CLiP - Catheter and Line Position Challenge.
DATA_ROOT = "/content/ranzcr-clip-catheter-line-classification"
RUN_ROOT = RUNS_ROOT / "ranzcr_clip"

# OOF comparison -> MLE ensemble reconstruction -> test prediction ->
# sample_submission validation -> optional Kaggle API submission.
result = run_complete_benchmark("ranzcr_clip")
pd.read_csv(RUN_ROOT / "comparison.csv") if (RUN_ROOT / "comparison.csv").is_file() else result


RANZCR exact resize cache:   0%|          | 0/33665 [00:00<?, ?it/s]

{'cached_images': 33665, 'new_files': 33665, 'pixel_checks': 10}
START seed=13 phase=baseline model=resnet18


model.safetensors:   0%|          | 0.00/46.8M [00:00<?, ?B/s]

  fit=1 epoch=1/40 loss=0.57112 inner_val_loss=0.50720 best_epoch=1 seconds=3.4
  fit=1 epoch=2/40 loss=0.38930 inner_val_loss=0.35711 best_epoch=2 seconds=1.1
  fit=1 epoch=3/40 loss=0.31350 inner_val_loss=0.32497 best_epoch=3 seconds=1.1
  fit=1 epoch=4/40 loss=0.27911 inner_val_loss=0.30585 best_epoch=4 seconds=1.1
  fit=1 epoch=5/40 loss=0.25945 inner_val_loss=0.29468 best_epoch=5 seconds=1.2
  fit=1 epoch=6/40 loss=0.24347 inner_val_loss=0.28844 best_epoch=6 seconds=1.1
  fit=1 epoch=7/40 loss=0.23343 inner_val_loss=0.29056 best_epoch=6 seconds=1.2
  fit=1 epoch=8/40 loss=0.22236 inner_val_loss=0.28510 best_epoch=8 seconds=1.1
  fit=1 epoch=9/40 loss=0.21312 inner_val_loss=0.28772 best_epoch=8 seconds=1.2
  fit=1 epoch=10/40 loss=0.20279 inner_val_loss=0.28378 best_epoch=10 seconds=1.1
  fit=1 epoch=11/40 loss=0.19471 inner_val_loss=0.28929 best_epoch=10 seconds=1.1
  fit=1 epoch=12/40 loss=0.18583 inner_val_loss=0.29147 best_epoch=10 seconds=1.1
  fit=1 epoch=13/40 loss=0.17262 i

model.safetensors:   0%|          | 0.00/21.4M [00:00<?, ?B/s]

...

median       mean  min  max  count
phase            model                                              
baseline         resnet18           11.0  11.533333    9   16     15
final_full_data  resnet18            5.0   4.666667    3    6      3
initial          efficientnet_b0     4.0   4.133333    3    5     15
                 resnet18           11.0  11.533333    9   16     15
initial_selected resnet18           11.0  11.533333    9   16     15
refined_selected resnet18           11.0  11.533333    9   16     15
refinement       efficientnet_b0     4.0   4.133333    3    5     15

Epoch-selection evidence: /content/drive/MyDrive/mlestar-runs/ranzcr_clip/selected_epochs.csv
Validated ranzcr_clip submission: rows=3582, columns=12


,StudyInstanceUID,ETT - Abnormal,ETT - Borderline,ETT - Normal,NGT - Abnormal,NGT - Borderline,NGT - Incompletely Imaged,NGT - Normal,CVC - Abnormal,CVC - Borderline,CVC - Normal,Swan Ganz Catheter Present
0,1.2.826.0.1.3680043.8.498.46923145579096002617...,0.025312,0.353841,0.695150,0.034737,0.069845,0.080650,0.789225,0.163778,0.371338,0.708984,0.420756
1,1.2.826.0.1.3680043.8.498.84006870182611080091...,0.000045,0.000075,0.000257,0.000273,0.000545,0.000601,0.000276,0.030497,0.061867,0.912923,0.000066
2,1.2.826.0.1.3680043.8.498.12219033294413119947...,0.000289,0.000476,0.000823,0.001105,0.002610,0.001173,0.004148,0.043645,0.184296,0.823893,0.000485
3,1.2.826.0.1.3680043.8.498.84994474380235968109...,0.018017,0.174764,0.820801,0.064283,0.040147,0.745117,0.093079,0.142232,0.392537,0.659342,0.112111
4,1.2.826.0.1.3680043.8.498.35798987793805669662...,0.000168,0.000264,0.000288,0.000671,0.001419,0.000264,0.002295,0.037421,0.128830,0.863281,0.000200


{
  "requested": true,
  "accepted": true,
  "status": "assets_ready",
  "asset_dir": "/content/drive/MyDrive/mlestar-runs/ranzcr_clip/kaggle_assets",
  "asset_zip": "/content/drive/MyDrive/mlestar-runs/MLE_STAR_RANZCR_Kaggle_Assets.zip",
  "asset_zip_mb": 129.19363117218018,
  "manifest": "/content/drive/MyDrive/mlestar-runs/ranzcr_clip/kaggle_assets/mle_star_ranzcr_manifest.json",
  "model_files": [
    "resnet18_seed_13.pt",
    "resnet18_seed_29.pt",
    "resnet18_seed_47.pt"
  ],
  "next_step": "Upload this ZIP as a private Kaggle Dataset, attach it and the RANZCR competition data to MLE_STAR_RANZCR_CLiP_Kaggle_Inference.ipynb, then Save Version -> Save & Run All -> Submit."
}
{
  "benchmark": "ranzcr_clip",
  "success": true,
  "submission_path": "/content/drive/MyDrive/mlestar-runs/ranzcr_clip/submission.csv",
  "latest_submission": null,
  "error": null
}


,seed,arm,metric_value
0,13,baseline,0.716431
1,13,mlestar_initial,0.716431
2,13,mlestar_refined,0.716431
3,13,mlestar_ensemble,0.716431
4,29,baseline,0.715289
5,29,mlestar_initial,0.715289
6,29,mlestar_refined,0.715289
7,29,mlestar_ensemble,0.715289
8,47,baseline,0.710502
9,47,mlestar_initial,0.710502


In [9]:
# Compact MLE-STAR experiment and Kaggle-asset table.
summary = pd.DataFrame([
    {
        "benchmark": name,
        "oof_epoch_protocol": value.get("oof_epoch_protocol"),
        "kaggle_asset_status": value.get("kaggle", {}).get("status"),
        "asset_zip": value.get("kaggle", {}).get("asset_zip"),
        "asset_zip_mb": value.get("kaggle", {}).get("asset_zip_mb"),
        "success": value.get("success"),
        "error": value.get("error"),
    }
    for name, value in PIPELINE_RESULTS.items()
])
display(summary)


,benchmark,oof_epoch_protocol,kaggle_asset_status,asset_zip,asset_zip_mb,success,error
0,ranzcr_clip,mle_inner_selected_epochs,assets_ready,/content/drive/MyDrive/mlestar-runs/MLE_STAR_R...,129.193631,True,None


The notebook keeps OOF comparison and Kaggle hidden-test submission separate.

- `comparison.csv` contains baseline and MLE-STAR OOF results using mean ROC-AUC.
- `submission.csv` is the public-test audit prediction in the exact sample schema.
- `MLE_STAR_RANZCR_Kaggle_Assets.zip` contains the exact final checkpoints,
  per-seed MLE ensemble weights and inference contract for the companion Kaggle Notebook.
- Upload the ZIP as a **private Kaggle Dataset**. In Kaggle, import
  `MLE_STAR_RANZCR_CLiP_Kaggle_Inference.ipynb`, attach both the official competition
  input and that private Dataset, select a compatible GPU, keep Internet off, and use
  **Save Version -> Save & Run All -> Submit**.
- OOF artifacts, epoch choices, final checkpoints and submissions remain stored under
  Google Drive when enabled, allowing completed stages to resume after disconnects.
- The local PNG cache changes no model input pixels; it only avoids repeated JPEG
  decode and resize work during the many OOF and final-model passes.
